# P18 · U2: Transformer vs LSTM
Runtime ▸ **T4 GPU**, then Run all (about 10 min). Best: run in the *same* Colab session as the transformer training.

In [ ]:
import os, zipfile, shutil, glob, subprocess, sys
for d in ("src", "out", "data"): os.makedirs(d, exist_ok=True)
if not os.path.exists("src/model.py"):
    from google.colab import files
    files.upload()                      # choose p18_repo.zip
    zipfile.ZipFile("p18_repo.zip").extractall(".")
    for f in glob.glob("p18repo/src/*.py"): shutil.copy(f, "src/")
    for f in glob.glob("p18repo/out/*"): shutil.copy(f, "out/")
print("src:", os.listdir("src"), "| out:", os.listdir("out"))

## Data check

In [ ]:
# Same splits as the transformer run. If data/ is gone (Colab session reset), rebuild it AND retrain the
# transformer on the new split, so the comparison stays fair (no train/test leakage).
env = {**os.environ, "PYTHONPATH": "src"}
if os.path.exists("data/train.txt"):
    print("Using existing data/ splits from the transformer run.")
else:
    print("data/ missing: rebuilding data and retraining the transformer (about 6 min) ...")
    subprocess.run([sys.executable, "src/data.py"], check=True, env=env)
    subprocess.run([sys.executable, "src/train.py"], check=True, env=env)
print({k: len(open(f"data/{k}.txt").read().split("\n")) for k in ("train", "val", "test")})

## Train the LSTM (same data, same 6 epochs, about 1.8M params)

In [ ]:
import sys, time, math, json, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, "src")
from types import SimpleNamespace
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from train import load, build_vocab, encode, evaluate

class CharLSTM(nn.Module):
    """2-layer character LSTM with the same interface as MiniGPT, so sample.generate() works on it."""
    def __init__(self, vocab, emb=128, hid=360, layers=2, dropout=0.1, block=32):
        super().__init__()
        self.c = SimpleNamespace(block_size=block)
        self.emb = nn.Embedding(vocab, emb)
        self.lstm = nn.LSTM(emb, hid, layers, batch_first=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hid, vocab)
    def forward(self, idx, targets=None):
        h, _ = self.lstm(self.drop(self.emb(idx)))
        logits = self.head(self.drop(h))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss
    def n_params(self):
        return sum(p.numel() for p in self.parameters())

dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
tr, va, te = load("data/train.txt"), load("data/val.txt"), load("data/test.txt")
stoi = build_vocab(tr + va + te)
xt, yt = encode(tr, stoi); xv, yv = encode(va, stoi); xe, ye = encode(te, stoi)
model = CharLSTM(len(stoi) + 2).to(dev)
print(f"device={dev} LSTM params={model.n_params():,} (transformer: 1,785,408)")

epochs, bs, lr = 6, 256, 2e-3            # same budget as the transformer
opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.01)
total = epochs * math.ceil(len(xt) / bs); warm = 200
sched = lambda s: lr * (s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
steps, tr_l, va_l, best, step, run, t0 = [], [], [], 1e9, 0, 0.0, time.time()
for ep in range(epochs):
    perm = torch.randperm(len(xt))
    for i in range(0, len(xt), bs):
        ix = perm[i:i+bs]; xb, yb = xt[ix].to(dev), yt[ix].to(dev)
        for g in opt.param_groups: g["lr"] = sched(step)
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        run = 0.9 * run + 0.1 * loss.item() if step else loss.item(); step += 1
        if step % 200 == 0 or step == total:
            v = evaluate(model, xv, yv, dev)
            steps.append(step); tr_l.append(run); va_l.append(v)
            print(f"ep {ep+1} step {step}/{total} train {run:.3f} val {v:.3f} ({v/math.log(2):.3f} bpc)")
            if v < best:
                best = v
                torch.save({"model": model.state_dict(), "stoi": stoi}, "out/lstm.pt")
secs = time.time() - t0
model.load_state_dict(torch.load("out/lstm.pt", map_location=dev)["model"])
test = evaluate(model, xe, ye, dev)
lstm_metrics = {"params": model.n_params(), "train_seconds": round(secs, 1),
                "val_bpc": best / math.log(2), "test_bpc": test / math.log(2)}
json.dump(lstm_metrics, open("out/metrics_lstm.json", "w"), indent=2); print(lstm_metrics)
plt.figure(figsize=(6, 4)); plt.plot(steps, tr_l, label="train (smoothed)"); plt.plot(steps, va_l, label="validation")
plt.xlabel("step"); plt.ylabel("loss (nats/char)"); plt.title("Char-LSTM training loss")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("out/lstm_loss_curve.png", dpi=150)

## Compare both models: loss, uniqueness, novelty, latency, sample names

In [ ]:
import pandas as pd
from sample import load_model, generate

known = {l.strip() for l in open("data/train.txt") if l.strip()}
itos = {i: c for c, i in stoi.items()}
tm, t_stoi, t_itos = load_model("out/mini_gpt.pt")
# re-score the transformer on the CURRENT test split so both models are measured identically
tm_test = evaluate(tm, xe, ye, "cpu") / math.log(2)
tm_val = evaluate(tm, xv, yv, "cpu") / math.log(2)
tmet = json.load(open("out/metrics_transformer.json"))
lstm = model.cpu().eval()

def gen_stats(m, s, i):
    names = [n for n in generate(m, s, i, n=1000, temperature=0.8, seed=0) if n]
    uniq = set(names); novel = [n for n in uniq if n.lower() not in known]
    t = time.time()
    for k in range(20): generate(m, s, i, n=1, temperature=0.8, seed=k)
    return {"unique_%": 100 * len(uniq) / len(names), "novel_% (of unique)": 100 * len(novel) / max(1, len(uniq)),
            "cpu_ms_per_name": (time.time() - t) / 20 * 1000}

rows = {
  "Mini-Transformer": {"params": tmet["params"], "val_bpc": tm_val, "test_bpc": tm_test,
                       "train_s": tmet["train_seconds"], **gen_stats(tm, t_stoi, t_itos)},
  "Char-LSTM":        {"params": lstm_metrics["params"], "val_bpc": lstm_metrics["val_bpc"], "test_bpc": lstm_metrics["test_bpc"],
                       "train_s": lstm_metrics["train_seconds"], **gen_stats(lstm, stoi, itos)},
}
table = pd.DataFrame(rows).T.round(3)
table.to_csv("out/comparison_table.csv"); print(table.to_string())
for name, (m, s, i) in {"Transformer": (tm, t_stoi, t_itos), "LSTM": (lstm, stoi, itos)}.items():
    print(f"\n{name} @T=0.8 seed=42:", generate(m, s, i, n=10, temperature=0.8, seed=42))
    print(f"{name} @T=1.2 seed=42:", generate(m, s, i, n=10, temperature=1.2, seed=42))

## Download

In [ ]:
!zip -r p18_lstm_results.zip out/lstm.pt out/lstm_loss_curve.png out/metrics_lstm.json out/comparison_table.csv
from google.colab import files; files.download("p18_lstm_results.zip")